# Tugas Manipulasi Data dengan Pandas
Dataset: `paket_a_transaksi_pos_2024_2025.csv`

Isi tugas: Task 1 (Indexing & Selection), Task 2 (Aritmatika), Task 3 (Filtering), Task 4 (Sorting), Task 5 (Grouping & Agregasi).

In [1]:
import pandas as pd

pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.width = 200

## Membaca data

In [2]:
df = pd.read_csv('../data/raw/paket_a_transaksi_pos_2024_2025.csv')
print(df.shape)
df.info()

(5030, 9)
<class 'pandas.DataFrame'>
RangeIndex: 5030 entries, 0 to 5029
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id_transaksi   5030 non-null   str  
 1   tanggal        5030 non-null   str  
 2   kode_gerai     5030 non-null   str  
 3   kode_kasir     4769 non-null   str  
 4   kode_produk    5030 non-null   str  
 5   jumlah         5030 non-null   int64
 6   harga_satuan   5030 non-null   str  
 7   diskon_persen  5030 non-null   int64
 8   status         5030 non-null   str  
dtypes: int64(2), str(7)
memory usage: 578.2 KB


---
# Task 1 — Indexing & Selection

### 1.1 Pembersihan format harga
Kolom `harga_satuan` terbaca sebagai objek. Selain koma desimal (`7500,00`), ada juga nilai berformat titik ribuan (`13.000`). Jika hanya mengganti koma menjadi titik, `13.000` akan terbaca sebagai `13.0` (salah). Maka titik ribuan dihapus dulu, baru koma diganti titik.

In [3]:
print('Tipe awal:', df['harga_satuan'].dtype)
print('Contoh nilai berformat aneh:', df.loc[~df['harga_satuan'].astype(str).str.fullmatch(r'\d+'), 'harga_satuan'].head(6).tolist())

h = df['harga_satuan'].astype(str).str.strip()
h = h.str.replace(r'^(\d{1,3})\.(\d{3})$', r'\1\2', regex=True)   # 13.000 -> 13000
h = h.str.replace(',', '.', regex=False)                            # 7500,00 -> 7500.00
df['harga_satuan'] = h.astype(float)

print('Tipe akhir:', df['harga_satuan'].dtype)
df['harga_satuan'].describe()

Tipe awal: str
Contoh nilai berformat aneh: ['7500,00', '11000,00', '13.000', '27000,00', '13500,00', '28.000']
Tipe akhir: float64


count    5,030.00
mean    22,185.09
std     15,205.70
min      4,000.00
25%     11,000.00
50%     16,000.00
75%     32,000.00
max     54,000.00
Name: harga_satuan, dtype: float64

### 1.2 Kuantitas negatif → positif (Boolean Indexing + `.loc` + `.abs()`)

In [4]:
print('Jumlah negatif sebelum:', (df['jumlah'] < 0).sum())
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()
print('Jumlah negatif sesudah :', (df['jumlah'] < 0).sum())

Jumlah negatif sebelum: 41
Jumlah negatif sesudah : 0


### 1.3 Imputasi `kode_kasir` kosong → `'K_UNKNOWN'`

In [5]:
print('NaN sebelum:', df['kode_kasir'].isna().sum())
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'
print('NaN sesudah :', df['kode_kasir'].isna().sum())

NaN sebelum: 261
NaN sesudah : 0


### 1.4 Seleksi posisi dengan `.iloc[]`

In [6]:
kolom = ['id_transaksi', 'tanggal', 'kode_gerai', 'jumlah']
idx_kolom = [df.columns.get_loc(c) for c in kolom]
df.iloc[:5, idx_kolom]

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


---
# Task 2 — Operasi Aritmatika

In [7]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11   # + PPN 11%

df[['id_transaksi', 'harga_satuan', 'jumlah', 'diskon_persen',
    'Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head()

,id_transaksi,harga_satuan,jumlah,diskon_persen,Total_Kotor,Nominal_Diskon,Total_Bersih
0,TRX0241065,"13,000.00",4,0,"52,000.00",0.00,"57,720.00"
1,TRX0240510,"21,000.00",1,0,"21,000.00",0.00,"23,310.00"
2,TRX0244483,"7,500.00",3,0,"22,500.00",0.00,"24,975.00"
3,TRX0242532,"6,500.00",2,0,"13,000.00",0.00,"14,430.00"
4,TRX0243509,"6,000.00",1,0,"6,000.00",0.00,"6,660.00"


---
# Task 3 — Filtering

### 3.1 Filter multi-kondisi: SELESAI, gerai G05 atau G09, Total_Bersih > Rp 100.000

In [8]:
filter_1 = df[(df['status'] == 'SELESAI')
              & (df['kode_gerai'].isin(['G05', 'G09']))
              & (df['Total_Bersih'] > 100_000)]
print('Jumlah baris yang memenuhi kriteria:', len(filter_1))
filter_1.head()

Jumlah baris yang memenuhi kriteria: 140


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,"47,500.00",10,SELESAI,"190,000.00","19,000.00","189,810.00"
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,"30,500.00",0,SELESAI,"122,000.00",0.00,"135,420.00"
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,"28,500.00",0,SELESAI,"114,000.00",0.00,"126,540.00"
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,"46,500.00",0,SELESAI,"139,500.00",0.00,"154,845.00"
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,"20,500.00",0,SELESAI,"123,000.00",0.00,"136,530.00"


### 3.2 Filter dengan `.query()`: gerai G10, jumlah ≥ 5

In [9]:
filter_2 = df.query("kode_gerai == 'G10' and jumlah >= 5")
print('Jumlah baris:', len(filter_2))
filter_2.head()

Jumlah baris: 39


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,"45,500.00",0,SELESAI,"546,000.00",0.00,"606,060.00"
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,"13,000.00",0,SELESAI,"78,000.00",0.00,"86,580.00"
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,"26,000.00",0,SELESAI,"312,000.00",0.00,"346,320.00"
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,"5,500.00",5,SELESAI,"66,000.00","3,300.00","69,597.00"
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,"47,000.00",0,SELESAI,"282,000.00",0.00,"313,020.00"


---
# Task 4 — Sorting

### 4.1 Urut `kode_gerai` A–Z, lalu `Total_Bersih` tertinggi ke terendah

In [10]:
df_urut = df.sort_values(['kode_gerai', 'Total_Bersih'], ascending=[True, False])
df_urut[['id_transaksi', 'kode_gerai', 'Total_Bersih']].head(10)

,id_transaksi,kode_gerai,Total_Bersih
244,TRX0243257,G01,"42,020,160.00"
2956,TRX0241367,G01,"672,660.00"
3588,TRX0240608,G01,"626,040.00"
2937,TRX0240631,G01,"559,440.00"
5,TRX0240658,G01,"545,454.00"
3889,TRX0241537,G01,"359,640.00"
2179,TRX0244635,G01,"349,650.00"
4270,TRX0240428,G01,"323,010.00"
695,TRX0244706,G01,"313,020.00"
4404,TRX0242179,G01,"313,020.00"


### 4.2 Top-5 transaksi dengan `.nlargest()`

In [11]:
top5 = df.nlargest(5, 'Total_Bersih')
top5[['id_transaksi', 'kode_gerai', 'jumlah', 'harga_satuan', 'diskon_persen', 'status', 'Total_Bersih']]

,id_transaksi,kode_gerai,jumlah,harga_satuan,diskon_persen,status,Total_Bersih
994,TRX0241002,G11,2093,"48,000.00",10,SELESAI,"100,363,536.00"
4985,TRX0244221,G09,1301,"51,500.00",10,SELESAI,"66,934,498.50"
467,TRX0243696,G06,892,"54,000.00",0,SELESAI,"53,466,480.00"
3393,TRX0241029,G07,1983,"23,000.00",10,SELESAI,"45,563,391.00"
244,TRX0243257,G01,1456,"32,500.00",20,SELESAI,"42,020,160.00"


> **Catatan:** nilai Top-5 di atas sangat besar karena ada transaksi dengan `jumlah` tidak wajar (ratusan sampai ribuan unit, padahal jumlah normal hanya 1–12). Tugas ini tidak meminta membuang data tersebut sehingga tetap ikut dihitung. Pada analisis pendapatan sebelumnya (`analisis_pendapatan_pos.ipynb`) baris seperti ini dikeluarkan.

---
# Task 5 — Grouping & Agregasi

### 5.1 Laporan agregasi per gerai (hanya status SELESAI)

In [12]:
df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = (df_selesai
    .groupby('kode_gerai')
    .agg(Jumlah_Transaksi=('id_transaksi', 'count'),
         Total_Unit_Terjual=('jumlah', 'sum'),
         Total_Omzet_Bersih=('Total_Bersih', 'sum'),
         Rata_Rata_Transaksi=('Total_Bersih', 'mean'))
    .sort_values('Total_Omzet_Bersih', ascending=False)
    .round(2))
laporan_gerai

,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,"125,638,152.75","315,673.75"
G09,379,3027,"101,157,630.00","266,906.68"
G06,383,3103,"91,228,624.50","238,194.84"
G07,384,4284,"80,559,748.50","209,791.01"
G01,450,2635,"70,967,378.25","157,705.28"
G03,372,3274,"59,991,226.50","161,266.74"
G10,401,1742,"58,964,892.75","147,044.62"
G08,408,1700,"35,630,084.25","87,328.64"
G12,406,1010,"24,699,220.50","60,835.52"


### 5.2 Transformasi kelompok: `Kontribusi_Gerai_Persen`
Total omzet gerai dihitung dari seluruh baris `df` (DataFrame utama) per `kode_gerai`.

In [13]:
total_gerai = df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
df['Kontribusi_Gerai_Persen'] = df['Total_Bersih'] / total_gerai * 100

# Cek: kontribusi seluruh transaksi dalam satu gerai harus berjumlah 100
print(df.groupby('kode_gerai')['Kontribusi_Gerai_Persen'].sum().round(6).unique())
df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head()

[100.]


,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,"57,720.00",0.06
1,TRX0240510,G09,"23,310.00",0.02
2,TRX0244483,G11,"24,975.00",0.02
3,TRX0242532,G05,"14,430.00",0.06
4,TRX0243509,G05,"6,660.00",0.03


---
## Menyimpan hasil

In [14]:
laporan_gerai.to_csv('../output/laporan_gerai.csv')
top5.to_csv('../output/top5_transaksi.csv', index=False)
df.to_csv('../data/processed/transaksi_tugas_pandas.csv', index=False)
print('Tersimpan di folder output/ dan data/processed/')

Tersimpan di folder output/ dan data/processed/
